# OmniVoice 快速开始

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k2-fsa/OmniVoice/blob/master/docs/OmniVoice.ipynb)

本 Notebook 演示 [OmniVoice](https://github.com/k2-fsa/OmniVoice) 的基础用法。OmniVoice 是一个支持 600+ 种语言的大规模多语言 zero-shot TTS（零样本文本转语音）模型。

**内容：**
1. 安装
2. 选项 A — Gradio Demo（交互式 Web UI，无需写代码）
3. 选项 B — Python API
   - 3.1 加载模型
   - 3.2 声音克隆
   - 3.3 声音设计
   - 3.4 自动声音


## 1. 安装

Colab 已经提供兼容的 PyTorch + CUDA 环境，因此这里只需要安装 OmniVoice。


In [ ]:
!pip install omnivoice

## 2. 选项 A — Gradio Demo

启动带公开 Gradio 链接的交互式 Web UI。`--share` 参数会创建一个临时公开 URL，方便你从任意浏览器访问 demo。

> **如果你更希望直接使用 Python API，请跳到下面的选项 B。**


In [ ]:
!omnivoice-demo --share

## 3. 选项 B — Python API

### 3.1 加载模型


In [ ]:
from omnivoice import OmniVoice
import soundfile as sf
import torch
from IPython.display import Audio, display

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
    load_asr=True,
)

### 3.2 声音克隆

从一段较短的参考音频片段（3-10 秒）克隆声音。你可以上传自己的 `ref.wav`，也可以使用任意音频文件。

`ref_text` 是可选项；如果省略，模型会使用 Whisper ASR 自动转写参考音频。


In [ ]:
from google.colab import files

print("Upload a reference audio file (wav/mp3/flac):")
uploaded = files.upload()
ref_audio_path = list(uploaded.keys())[0]
print(f"Uploaded: {ref_audio_path}")

In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice cloning.",
    ref_audio=ref_audio_path,
    # ref_text="Transcription of the reference audio.",  # optional
)

sf.write("clone_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.3 声音设计

通过 speaker attributes（说话人属性）描述想要的声音，不需要参考音频。

支持的属性包括：性别、年龄、音高、风格（耳语）、英语口音、中文方言。完整列表见 [docs/voice-design_zh.md](https://github.com/k2-fsa/OmniVoice/blob/master/docs/voice-design_zh.md)。


In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice design.",
    instruct="female, low pitch, british accent",
)

sf.write("design_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.4 自动声音

让模型自动选择声音；不需要参考音频，也不需要 instruct。


In [ ]:
audio = model.generate(
    text="This is a sentence generated with automatic voice selection.",
)

sf.write("auto_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))